# Recomendation System

Lo primero de todo es cargar el dataset de *Book-Crossing* (`BX-Book-Ratings.csv` y `BX-Users.csv`).

In [1]:
import pandas as pd

ratings_path = "./datasets/BX-Book-Ratings.csv"
users_path = "./datasets/BX-Users.csv"

ratings = pd.read_csv(ratings_path, sep=';', encoding='latin-1', on_bad_lines='skip')
users = pd.read_csv(users_path, sep=';', encoding='latin-1', on_bad_lines='skip')

Aplicamos una limpieza inicial eliminando espacios en blanco ocultos en los nombres de las columnas para evitar errores de indexación más adelante.

In [2]:
ratings.columns = ratings.columns.str.strip()
users.columns = users.columns.str.strip()

### Segmentación por Edad
Realizamos un filtrado inicial para eliminar usuarios con edades inválidas o nulas (quedándonos solo con el rango coherente de 10 a 90 años). Luego, creamos la segmentación para analizar la equidad del modelo dividiendo a la población en dos grupos lógicos: **Jóvenes** ($\le 35$ años) y **Adultos** ($> 35$ años).

In [3]:
# Filtramos primero el rango coherente de edad
users_v = users[(users['Age'] >= 10) & (users['Age'] <= 90)].copy()

# Creamos la separación de los dos grupos lógicos
users_v['Age_Group'] = users_v['Age'].apply(lambda x: 'Joven' if x <= 35 else 'Adulto')
users_v['Age_Group'].value_counts()

Age_Group
Joven     98206
Adulto    68391
Name: count, dtype: int64

### Segmentación Geográfica
Procesamos la columna `Location` para extraer de forma limpia el país de residencia de cada usuario. Segmentamos la muestra dividiendo entre la región con mayor volumen de datos históricos (**USA y Canadá**) frente al **Resto del Mundo**, con el fin de evaluar si el algoritmo penaliza a las regiones minoritarias en el dataset.

In [4]:
users_v['Country'] = users_v['Location'].apply(lambda x: str(x).split(',')[-1].strip().lower())
users_v['Is_USA_or_Can'] = users_v['Country'].isin(['usa', 'canada'])
users_v['Is_USA_or_Can'].value_counts()

Is_USA_or_Can
True     87224
False    79373
Name: count, dtype: int64

### Segmentación por Comportamiento de Consumo
Calculamos el volumen total de interacciones que tiene cada libro para determinar el umbral del top 20% más votado (libros comerciales). Luego, promediamos el tipo de lectura de cada usuario para etiquetarlos en dos perfiles de comportamiento: **Usuarios Mainstream** (consumidores de éxitos populares) y **Usuarios de Nicho** (lectores de material menos comercial).

In [5]:
# 1. Identificar qué libros son populares (Top 20% más votados)
libro_counts = ratings['ISBN'].value_counts()
umbral_popularidad = libro_counts.quantile(0.80)

# 2. Etiquetar cada interacción como popular (1) o no popular (0)
ratings_pop = ratings.merge(libro_counts.to_frame('Votos_Libro'), left_on='ISBN', right_index=True)
ratings_pop['es_popular'] = (ratings_pop['Votos_Libro'] >= umbral_popularidad).astype(int)

# 3. Calcular el porcentaje/proporción de lecturas populares por usuario
user_ratio_popular = ratings_pop.groupby('User-ID')['es_popular'].mean()

# 4. Establecer un criterio claro de corte (ej. si más del 50% de lo que lee es popular)
usuarios_mainstream = user_ratio_popular[user_ratio_popular >= 0.80].index
usuarios_nicho = user_ratio_popular[user_ratio_popular < 0.20].index
print(f"--> Usuarios Mainstream identificados: {len(usuarios_mainstream)}")

--> Usuarios Mainstream identificados: 67982


# 6. Entrenamiento del Algoritmo de Recomendación SVD
### Nota de Arquitectura: ¿Por qué la segmentación y limpieza van ANTES del entrenamiento?
Es una regla fundamental en Machine Learning (*Garbage In, Garbage Out*). Necesitamos estructurar y limpiar los datos de los usuarios primero para:
1. **Evitar datos sucios:** No gastar recursos entrenando a la IA con perfiles eliminados (ej. usuarios con edades de 0 o 200 años).
2. **Cruzar datasets eficientemente:** Realizar un filtrado (usuarios con $\ge 8$ ratings) sobre los datos ya validados antes de alimentar al algoritmo.

Con los datos limpios, aplicamos un **Filtrado Colaborativo basado en Modelo (SVD)**, utilizando un split del 80% para entrenamiento y 20% para testeo.

In [7]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

print("--> Entrenando el modelo de recomendación...")
# Unimos tablas y filtramos usuarios con al menos 8 ratings para agilizar la compu
df_completo = pd.merge(ratings, users_v, on='User-ID', how='inner')
user_counts = df_completo['User-ID'].value_counts()
df_completo = df_completo[df_completo['User-ID'].isin(user_counts[user_counts >= 8].index)]

# Cambiamos nombres para la librería Surprise
df_surprise = df_completo[['User-ID', 'ISBN', 'Book-Rating']].copy()
df_surprise.columns = ['userID', 'itemID', 'rating']

reader = Reader(rating_scale=(0, 10))
data = Dataset.load_from_df(df_surprise, reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Ejecutamos el SVD
algo = SVD(random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)

# Guardamos todo cruzado con tus etiquetas para la celda de evaluación
df_preds = pd.DataFrame(predictions, columns=['uid', 'iid', 'r_ui', 'est', 'details'])
df_preds = df_preds.merge(users_v[['User-ID', 'Age', 'Is_USA_or_Can']], left_on='uid', right_on='User-ID', how='inner')
df_preds['Es_Mainstream'] = df_preds['uid'].isin(usuarios_mainstream)
print("--> Algoritmo SVD entrenado y predicciones listas.")

ImportError: cannot import name 'similarities' from partially initialized module 'surprise' (most likely due to a circular import) (c:\Users\ivant\Downloads\Recomendation-System\.venv\Lib\site-packages\surprise\__init__.py)

# 7. Evaluación Final de Fairness (Métricas de Error RMSE)
Calculamos y desglosamos la métrica **RMSE (Root Mean Squared Error)** de forma aislada para cada uno de los subgrupos creados. Comparando la diferencia de error absoluto entre Jóvenes/Adultos, USA-Canadá/Resto del Mundo y Mainstream/Nicho, determinaremos si el algoritmo SVD presenta sesgos algorítmicos o si se comporta de manera equitativa (*fairness*).

In [ ]:
from surprise import accuracy

print("==========================================================")
print("🎯 EVALUACIÓN DE SESGOS DE ERROR (RMSE)")
print("==========================================================")

# Map de atributos por usuario para evitar búsquedas repetidas
user_attrs = df_preds.groupby('uid').first()[['Age', 'Is_USA_or_Can', 'Es_Mainstream']]
age_map = user_attrs['Age'].to_dict()
usa_map = user_attrs['Is_USA_or_Can'].to_dict()
main_map = user_attrs['Es_Mainstream'].to_dict()

p_jovenes = [p for p in predictions if age_map.get(p.uid, 999) <= 35]
p_adultos = [p for p in predictions if age_map.get(p.uid, 999) > 35]

p_usa = [p for p in predictions if usa_map.get(p.uid, False)]
p_resto = [p for p in predictions if not usa_map.get(p.uid, False)]

p_main = [p for p in predictions if main_map.get(p.uid, False)]
p_nicho = [p for p in predictions if not main_map.get(p.uid, False)]

print(f"[EDAD] RMSE Jóvenes: {accuracy.rmse(p_jovenes, verbose=False):.4f} | RMSE Adultos: {accuracy.rmse(p_adultos, verbose=False):.4f}")
print(f"[GEOGRAFÍA] RMSE USA/Can: {accuracy.rmse(p_usa, verbose=False):.4f} | RMSE Resto Mundo: {accuracy.rmse(p_resto, verbose=False):.4f}")
print(f"[GUSTOS] RMSE Mainstream: {accuracy.rmse(p_main, verbose=False):.4f} | RMSE Nicho: {accuracy.rmse(p_nicho, verbose=False):.4f}")

In [ ]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=5, threshold=7):
    """Devuelve la precisión y el recall por usuario a partir de un umbral de relevancia"""
    user_est_true = defaultdict(list)
    for uid, _, r_ui, est, _ in predictions:
        user_est_true[uid].append((est, r_ui))

    precisions = dict()
    recalls = dict()
    
    for uid, user_ratings in user_est_true.items():
        # Ordenamos de mayor a menor predicción estimada
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum((r_ui >= threshold) for (_, r_ui) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(((r_ui >= threshold) and (est >= threshold)) for (est, r_ui) in user_ratings[:k])

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls

print("--> Función de métricas a nivel de recomendación compilada.")

In [ ]:
user_precisions, user_recalls = precision_recall_at_k(predictions, k=50, threshold=7)

df_met_user = pd.DataFrame(
    [
        {
            'User-ID': uid,
            'Precision@5': prec,
            'Recall@5': user_recalls.get(uid, 0),
        }
        for uid, prec in user_precisions.items()
    ]
)

df_fair_met = df_met_user.merge(
    users_v[['User-ID', 'Age', 'Is_USA_or_Can']],
    on='User-ID',
    how='inner'
)
df_fair_met['Es_Mainstream'] = df_fair_met['User-ID'].isin(usuarios_mainstream)

summary = {
    'Jóvenes': df_fair_met[df_fair_met['Age'] <= 35][['Precision@5', 'Recall@5']].mean(),
    'Adultos': df_fair_met[df_fair_met['Age'] > 35][['Precision@5', 'Recall@5']].mean(),
    'USA_Can': df_fair_met[df_fair_met['Is_USA_or_Can']][['Precision@5', 'Recall@5']].mean(),
    'Resto': df_fair_met[~df_fair_met['Is_USA_or_Can']][['Precision@5', 'Recall@5']].mean(),
    'Mainstream': df_fair_met[df_fair_met['Es_Mainstream']][['Precision@5', 'Recall@5']].mean(),
    'Nicho': df_fair_met[~df_fair_met['Es_Mainstream']][['Precision@5', 'Recall@5']].mean(),
}

print("==========================================================")
print("🎯 RESUMEN DE PRECISIÓN@5 Y RECALL@5 POR GRUPO")
print("==========================================================")
print("Edad:")
print(f"- Jóvenes    : P@5={summary['Jóvenes']['Precision@5']:.4f}, R@5={summary['Jóvenes']['Recall@5']:.4f}")
print(f"- Adultos    : P@5={summary['Adultos']['Precision@5']:.4f}, R@5={summary['Adultos']['Recall@5']:.4f}")
print("Geografía:")
print(f"- USA/Can    : P@5={summary['USA_Can']['Precision@5']:.4f}, R@5={summary['USA_Can']['Recall@5']:.4f}")
print(f"- Resto Mundo: P@5={summary['Resto']['Precision@5']:.4f}, R@5={summary['Resto']['Recall@5']:.4f}")
print("Gustos:")
print(f"- Mainstream : P@5={summary['Mainstream']['Precision@5']:.4f}, R@5={summary['Mainstream']['Recall@5']:.4f}")
print(f"- Nicho      : P@5={summary['Nicho']['Precision@5']:.4f}, R@5={summary['Nicho']['Recall@5']:.4f}")
print("==========================================================")

In [ ]:
total_items = df_completo['ISBN'].nunique()

def calcular_cobertura(preds_grupo):
    items_recomendados = set()
    user_preds = defaultdict(list)
    for u, i, _, est, _ in preds_grupo:
        user_preds[u].append((i, est))
    for u, u_ratings in user_preds.items():
        u_ratings.sort(key=lambda x: x[1], reverse=True)
        for item, _ in u_ratings[:5]:
            items_recomendados.add(item)
    return (len(items_recomendados) / total_items) * 100

print("==========================================================")
print("📦 COBERTURA DEL CATÁLOGO (%) POR GRUPO")
print("==========================================================")
print(f"[EDAD]      Cobertura Jóvenes: {calcular_cobertura(p_jovenes):.2f}% | Adultos: {calcular_cobertura(p_adultos):.2f}%")
print(f"[GEOGRAFÍA] Cobertura USA/Can: {calcular_cobertura(p_usa):.2f}% | Resto Mundo: {calcular_cobertura(p_resto):.2f}%")
print(f"[GUSTOS]    Cobertura Mainstream: {calcular_cobertura(p_main):.2f}% | Nicho: {calcular_cobertura(p_nicho):.2f}%")
print("==========================================================")

# 11. Análisis de Significancia Estadística (Validación del Inciso 4)
Para demostrar si las diferencias de error encontradas entre los subgrupos son estadísticamente significativas (y no fruto del azar), aplicamos el **Test de Mann-Whitney U**. Este test no paramétrico compara las distribuciones del error absoluto ($|Valor Real - Valor Estimado|$) de cada par de grupos. Si el valor $p$ ($p\text{-value}$) es menor a $0.05$, se confirma científicamente la existencia de un sesgo algorítmico (*unfairness*).

In [ ]:
import scipy.stats as stats
import numpy as np

print("--> Calculando pruebas estadísticas de Fairness...")

# Creamos la columna de error absoluto para cada predicción individual
df_preds['Error_Absoluto'] = np.abs(df_preds['r_ui'] - df_preds['est'])

# 1. Separación de errores por Edad
err_jovenes = df_preds[df_preds['Age'] <= 35]['Error_Absoluto']
err_adultos = df_preds[df_preds['Age'] > 35]['Error_Absoluto']
stat_edad, p_edad = stats.mannwhitneyu(err_jovenes, err_adultos, alternative='two-sided')

# 2. Separación de errores por Geografía
err_usa = df_preds[df_preds['Is_USA_or_Can'] == True]['Error_Absoluto']
err_resto = df_preds[df_preds['Is_USA_or_Can'] == False]['Error_Absoluto']
stat_geo, p_geo = stats.mannwhitneyu(err_usa, err_resto, alternative='two-sided')

# 3. Separación de errores por Gustos
err_main = df_preds[df_preds['Es_Mainstream'] == True]['Error_Absoluto']
err_nicho = df_preds[df_preds['Es_Mainstream'] == False]['Error_Absoluto']
stat_gustos, p_gustos = stats.mannwhitneyu(err_main, err_nicho, alternative='two-sided')

print("==========================================================")
print("📊 RESULTADOS DE PRUEBAS ESTADÍSTICAS (MANN-WHITNEY U)")
print("==========================================================")
print(f"[EDAD]      p-value: {p_edad:.4e} | ¿Es significativo?: {'SÍ (Hay Sesgo)' if p_edad < 0.05 else 'NO (Es Equitativo)'}")
print(f"[GEOGRAFÍA] p-value: {p_geo:.4e} | ¿Es significativo?: {'SÍ (Hay Sesgo)' if p_geo < 0.05 else 'NO (Es Equitativo)'}")
print(f"[GUSTOS]    p-value: {p_gustos:.4e} | ¿Es significativo?: {'SÍ (Hay Sesgo)' if p_gustos < 0.05 else 'NO (Es Equitativo)'}")
print("==========================================================")

# 13. Experimento de Impacto: Modificación de la Partición Train/Test (Inciso 5)
Para cumplir con el inciso 5, exploramos el impacto de modificar la partición de datos. Reducimos el set de entrenamiento al 50% (antes era 80%) y aumentamos el de testeo al 50% (`test_size=0.5`). Evaluamos cómo este cambio drástico en la disponibilidad de datos afecta el rendimiento general y si acentúa o mitiga las diferencias y sesgos entre los subgrupos.

In [ ]:
print("--> Ejecutando Experimento con nueva partición 50/50...")

# 1. Hacemos el nuevo split con test_size=0.5
trainset_50, testset_50 = train_test_split(data, test_size=0.5, random_state=42)

# 2. Entrenamos un nuevo modelo SVD
algo_50 = SVD(random_state=42)
algo_50.fit(trainset_50)
predictions_50 = algo_50.test(testset_50)

# 3. Guardamos las nuevas predicciones cruzadas con los grupos
df_preds_50 = pd.DataFrame(predictions_50, columns=['uid', 'iid', 'r_ui', 'est', 'details'])
df_preds_50 = df_preds_50.merge(users_v[['User-ID', 'Age', 'Is_USA_or_Can']], left_on='uid', right_on='User-ID', how='inner')
df_preds_50['Es_Mainstream'] = df_preds_50['uid'].isin(usuarios_mainstream)

print("==========================================================")
print("🎯 NUEVOS ERRORES (RMSE) CON PARTICIÓN 50/50")
print("==========================================================")
rmse_gen_50 = accuracy.rmse(predictions_50, verbose=False)
print(f"RMSE General (50/50): {rmse_gen_50:.4f}\n")

# Filtramos las nuevas sublistas para calcular los RMSE por grupo
p_jov_50 = [p for p in predictions_50 if df_preds_50.loc[df_preds_50['uid'] == p.uid, 'Age'].values[0] <= 35]
p_adu_50 = [p for p in predictions_50 if df_preds_50.loc[df_preds_50['uid'] == p.uid, 'Age'].values[0] > 35]
print(f"[EDAD]      RMSE Jóvenes: {accuracy.rmse(p_jov_50, verbose=False):.4f} | RMSE Adultos: {accuracy.rmse(p_adu_50, verbose=False):.4f}")

p_usa_50 = [p for p in predictions_50 if df_preds_50.loc[df_preds_50['uid'] == p.uid, 'Is_USA_or_Can'].values[0] == True]
p_res_50 = [p for p in predictions_50 if df_preds_50.loc[df_preds_50['uid'] == p.uid, 'Is_USA_or_Can'].values[0] == False]
print(f"[GEOGRAFÍA] RMSE USA/Can: {accuracy.rmse(p_usa_50, verbose=False):.4f} | RMSE Resto Mundo: {accuracy.rmse(p_res_50, verbose=False):.4f}")

p_main_50 = [p for p in predictions_50 if df_preds_50.loc[df_preds_50['uid'] == p.uid, 'Es_Mainstream'].values[0] == True]
p_nich_50 = [p for p in predictions_50 if df_preds_50.loc[df_preds_50['uid'] == p.uid, 'Es_Mainstream'].values[0] == False]
print(f"[GUSTOS]    RMSE Mainstream: {accuracy.rmse(p_main_50, verbose=False):.4f} | RMSE Nicho: {accuracy.rmse(p_nich_50, verbose=False):.4f}")
print("==========================================================")